# Approach 3: TaPas — Table-Aware BERT for Question Answering

**TaPas** (Table Parser) is a BERT variant pre-trained to reason over tables.

**Key innovation:** TaPas adds **row and column positional embeddings** on top of BERT's word embeddings. This lets the model learn the 2D structure of a table — it knows a cell belongs to column `salary` and row 3, not just that it's a number.

**Model:** `google/tapas-base-finetuned-wtq` (fine-tuned on WikiTableQuestions)  
**Architecture:** BERT-base + table-specific embeddings (~110M parameters)  
**Output:** Cell coordinates (row/column selections) + optional aggregation operator (SUM, AVG, COUNT, NONE)

In [ ]:
# !pip install transformers torch pandas

In [ ]:
import pandas as pd
from transformers import pipeline, TapasTokenizer, TapasForQuestionAnswering
import torch
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/employees.csv')
df.head()

## Step 1: Understand TaPas's unique input format

TaPas receives:
1. The **question** as text tokens
2. The **table** with special positional encodings for each cell's row + column index

The tokenizer automatically flattens the table while preserving this structure.

In [ ]:
MODEL_ID = "google/tapas-base-finetuned-wtq"

print(f"Loading {MODEL_ID}...")
tokenizer = TapasTokenizer.from_pretrained(MODEL_ID)
model = TapasForQuestionAnswering.from_pretrained(MODEL_ID)
model.eval()
print("Model loaded!")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# TaPas requires all table values to be strings
table = df.astype(str)

# Peek at the tokenized input to understand the structure
question = "Who has the highest salary?"

encoding = tokenizer(
    table=table.head(5),  # just first 5 rows for inspection
    queries=question,
    return_tensors='pt',
    padding=True,
    truncation=True
)

print("Encoding keys:", list(encoding.keys()))
print("\ninput_ids shape:", encoding['input_ids'].shape)
print("token_type_ids shape:", encoding['token_type_ids'].shape)
print("\ntoken_type_ids encodes [segment, column_id, row_id, prev_label, column_rank, inv_column_rank]:")
print(encoding['token_type_ids'][0, :15])  # first 15 tokens

## Step 2: Inference — Question → Cell Selection + Aggregation

In [ ]:
def ask_tapas(question: str, table: pd.DataFrame) -> dict:
    """Ask TaPas a question about a table.
    
    Returns:
        dict with 'answer', 'selected_cells', 'aggregation', and 'raw_values'
    """
    table_str = table.astype(str)
    
    encoding = tokenizer(
        table=table_str,
        queries=question,
        return_tensors='pt',
        padding=True,
        truncation=True
    )
    
    with torch.no_grad():
        outputs = model(**encoding)
    
    # Decode the predicted cells and aggregation
    predicted_answer_coordinates, predicted_aggregation = tokenizer.convert_logits_to_predictions(
        encoding,
        outputs.logits.detach(),
        outputs.logits_aggregation.detach()
    )
    
    aggregation_labels = {0: 'NONE', 1: 'SUM', 2: 'AVERAGE', 3: 'COUNT'}
    agg = aggregation_labels[predicted_aggregation[0]]
    
    # Extract values from selected cells
    coordinates = predicted_answer_coordinates[0]
    selected_cells = []
    raw_values = []
    
    for row_idx, col_idx in coordinates:
        cell_value = table.iloc[row_idx, col_idx]
        col_name = table.columns[col_idx]
        selected_cells.append(f"row {row_idx} / {col_name}")
        raw_values.append(cell_value)
    
    # Compute final answer
    if agg == 'COUNT':
        answer = str(len(raw_values))
    elif agg == 'SUM' and raw_values:
        answer = str(sum(float(v) for v in raw_values))
    elif agg == 'AVERAGE' and raw_values:
        answer = str(sum(float(v) for v in raw_values) / len(raw_values))
    else:
        answer = ', '.join(str(v) for v in raw_values)
    
    return {
        'answer': answer,
        'aggregation': agg,
        'selected_cells': selected_cells,
        'raw_values': raw_values
    }

## Step 3: Ask questions and inspect the model's reasoning

In [ ]:
questions = [
    "Who has the highest salary?",
    "How many employees work in Engineering?",
    "What is the average performance score?",
    "Which employee has the most years of experience?",
]

for q in questions:
    result = ask_tapas(q, df)
    print(f"Question: {q}")
    print(f"  Answer:      {result['answer']}")
    print(f"  Aggregation: {result['aggregation']}")
    print(f"  Cells used:  {result['selected_cells'][:3]}{'...' if len(result['selected_cells']) > 3 else ''}")
    print()

## Step 4: Visualize cell selection — what TaPas 'looks at'

In [ ]:
def highlight_selected(question: str, table: pd.DataFrame):
    """Show which cells TaPas selected to answer the question."""
    result = ask_tapas(question, table)
    
    # Parse coordinates from selected_cells
    table_str = table.astype(str)
    encoding = tokenizer(table=table_str, queries=question, return_tensors='pt', truncation=True)
    with torch.no_grad():
        outputs = model(**encoding)
    coords, _ = tokenizer.convert_logits_to_predictions(
        encoding, outputs.logits.detach(), outputs.logits_aggregation.detach()
    )
    
    selected_set = set(coords[0])
    
    print(f"Question: {question}")
    print(f"Answer: {result['answer']} (via {result['aggregation']})")
    print(f"\nSelected cells (highlighted with >>>):")
    print(" | ".join(f"{c:20s}" for c in table.columns))
    print("-" * (23 * len(table.columns)))
    for row_idx, row in table.head(10).iterrows():
        row_str = ""
        for col_idx, val in enumerate(row):
            marker = ">>>" if (row_idx, col_idx) in selected_set else "   "
            row_str += f"{marker}{str(val):17s} | "
        print(row_str)

highlight_selected("Who has the highest salary?", df)

## Step 5: TaPas architecture — what makes it table-aware

In [ ]:
print("TaPas Model Architecture:")
print("=" * 50)
for name, module in model.named_children():
    params = sum(p.numel() for p in module.parameters())
    print(f"  {name}: {params:>12,} parameters")

print()
print("TaPas-specific embeddings (vs standard BERT):")
tapas_embeddings = model.tapas.embeddings
for name, embed in tapas_embeddings.named_children():
    if hasattr(embed, 'weight'):
        print(f"  {name}: {embed.weight.shape}")

## Key Takeaways

| | TaPas |
|---|---|
| **Architecture** | BERT + row/column embeddings |
| **Output type** | Cell selection + aggregation operator |
| **Scale** | Limited by BERT context (~512 tokens ≈ ~50 rows) |
| **Accuracy** | ~57% on WikiTableQuestions (WTQ) |
| **Best for** | Lookup and aggregation Q&A over small-medium tables |

**What makes it special:** Unlike approach 1, TaPas actually **learned structural table semantics** during pre-training — it knows columns have meaning independent of their position.

**Next:** See `04_tapex_table_qa.ipynb` — Microsoft's TAPEX uses a generative approach (BART) and achieves higher accuracy.